<a href="https://colab.research.google.com/github/HERO-DS/Computational-Drug-Discovery-LRRK2-gene/blob/main/CDD_ML_Part_3_LRRK2_Descriptor_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Descriptor Calculation and Dataset Preparation**

we will be calculating molecular descriptors that are essentially quantitative description of the compounds in the dataset.

## **Load bioactivity data**

Download the curated ChEMBL bioactivity data that has been pre-processed from Parts 1 and 2 of this Bioinformatics Project series. Here we will be using the **bioactivity_data_3class_pIC50.csv** file that essentially contain the pIC50 values that we will be using for building a regression model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

# Define file path in Drive
file_path = Path('/content/drive/MyDrive/Colab Notebooks/data/LRRK2_04_bioactivity_data_3class_pIC50.csv')

# Read CSV file
df3 = pd.read_csv(file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df3

,molecule_chembl_id,canonical_smiles,class,MW,LogP,NumHDonors,NumHAcceptors,pIC50
0,CHEMBL1771409,Cc1cc(N/N=C/c2ccc(O)c(O)c2)nc2ccccc12,inactive,293.326,3.40042,3.0,5.0,4.879426
1,CHEMBL1771411,Cc1cc(N/N=C/c2ccncc2)nc2ccccc12,intermediate,262.316,3.38422,1.0,4.0,5.387216
2,CHEMBL1933288,C[C@@H]1CCNC(=O)c2cc3ccc(C(=O)Nc4nc5ccccc5n4CC...,active,458.566,3.88950,2.0,4.0,7.795880
3,CHEMBL2012582,COc1cc(C(=O)N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc2c...,active,570.698,3.43870,1.0,9.0,7.886057
4,CHEMBL509032,COc1cc(N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc(Cl)c(N...,active,614.216,5.02410,2.0,10.0,8.107905
...,...,...,...,...,...,...,...,...
3778,CHEMBL6192175,CC(C)n1ccc(Nc2nc(N(C)[C@@H](C)C3CC3)c3cc[nH]c3...,active,339.447,3.71360,2.0,5.0,6.080922
3779,CHEMBL6190367,Cn1cc(Nc2nc(N[C@H]3CCc4ccccc43)c3nc[nH]c3n2)cn1,active,346.398,2.92940,3.0,6.0,7.091515
3780,CHEMBL6190371,CC(C)n1cc(Nc2nc(N[C@H]3CCc4ccccc43)c3nc[nH]c3n...,active,374.452,3.97330,3.0,6.0,8.221849
3781,CHEMBL6191498,CC(C)n1cc(Nc2nc(N[C@@H](C)C3CC3)c3c(C#N)c[nH]c...,active,350.430,3.56098,3.0,6.0,8.221849


## **Calculate ECFP4 (Morgan Radius 2) Fingerprints**


In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 38.6 MB/s eta 0:00:00


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def generate_ecfp4_fingerprints(smiles_series, radius=2, n_bits=1024):
    fingerprints = []
    for smi in smiles_series:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
            fingerprints.append(np.array(fp))
        else:
            fingerprints.append(np.zeros((n_bits,)))
    return pd.DataFrame(fingerprints)

# Generate X matrix (Descriptor Features)
df3_X = generate_ecfp4_fingerprints(df3['canonical_smiles'])

[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerator
[04:57:26] DEPRECATION WARNING: please use MorganGenerat

## **Preparing the X and Y Data Matrices**

In [ ]:
df3_X

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3778,0,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,1,0,0
3779,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3780,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3781,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
# Extract target Y variable
df3_Y = df3['pIC50']
df3_Y

,pIC50
0,4.879426
1,5.387216
2,7.795880
3,7.886057
4,8.107905
...,...
3778,6.080922
3779,7.091515
3780,8.221849
3781,8.221849


## **Combining X and Y variable**

In [ ]:
# Combine X (descriptors) and Y (pIC50) into one final dataset
dataset3 = pd.concat([df3_X, df3_Y], axis=1)
dataset3

,0,1,2,3,4,5,6,7,8,9,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,pIC50
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4.879426
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5.387216
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,7.795880
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,7.886057
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,8.107905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3778,0,1,1,0,0,0,1,0,0,0,...,0,0,0,0,1,0,1,0,0,6.080922
3779,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,7.091515
3780,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,8.221849
3781,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,8.221849


# **Let's download the CSV file**

In [ ]:
# Save dataset directly to Google Drive for Part 4 (Model Building)
dataset3.to_csv('/content/drive/MyDrive/Colab Notebooks/data/LRRK2_06_bioactivity_data_3class_pIC50_ecfp4.csv', index=False)